# 4. SISTEM REKOMENDASI (V2)

**Jurnal: Ekstraksi Kata Kunci dengan N-Gram dan IndoBERT untuk Rekomendasi Wisata Bogor**

---

## Alur Sistem Rekomendasi:
- **Pencarian (Search):** Menggunakan **IndoBERT** (semantic matching)
- **Rekomendasi Detail:** Menggunakan **N-Gram + TF-IDF** (keyword matching)

```
                    ┌──────────────────────┐
Input Query ──────► │    INDOBERT (100%)   │ ──► HASIL PENCARIAN
  (Search)          │ (Semantic Matching)  │
                    └──────────────────────┘

                    ┌──────────────────────┐
Detail Wisata ────► │ N-GRAM + TF-IDF (100%) │ ──► REKOMENDASI
 (Related)          │  (Keyword Matching)  │     (Halaman Detail)
                    └──────────────────────┘
```

In [16]:
import pandas as pd
import numpy as np
import pickle
import torch
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import warnings
import re
warnings.filterwarnings('ignore')

DATA_PATH = './data/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("✅ Libraries imported!")

✅ Libraries imported!


In [17]:
# Load semua data yang dibutuhkan
print("🔄 Loading data...\n")

df = pd.read_csv(f'{DATA_PATH}data_with_keywords.csv')
tfidf_matrix = np.load(f'{DATA_PATH}tfidf_matrix.npy')
with open(f'{DATA_PATH}tfidf_vectorizer.pkl', 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
indobert_embeddings = np.load(f'{DATA_PATH}indobert_embeddings.npy')

# ========== PERBAIKAN SHAPE MISMATCH ==========
# Karena Notebook 03 melakukan filtering empty strings, jumlah baris IndoBERT (295) 
# mungkin tidak sama dengan TF-IDF (296) dari Notebook 02.
# Kita harus menyamakan keduanya dengan filter yang sama.
if len(df) != len(indobert_embeddings):
    print(f"⚠️ SHAPE MISMATCH DETECTED: DF={len(df)}, IndoBERT={len(indobert_embeddings)}")
    print("🔄 Aligning data by filtering empty descriptions (same as notebook 03)...")
    
    # Cari kolom teks yang relevan
    text_col = 'deskripsi_clean' if 'deskripsi_clean' in df.columns else 'deskripsi_ngram'
    
    # Buat mask (baris dengan deskripsi valid/tidak kosong)
    mask = df[text_col].fillna('').astype(str).str.strip() != ''
    
    # Apply mask ke DF dan TF-IDF agar sinkron dengan IndoBERT
    df = df[mask].reset_index(drop=True)
    tfidf_matrix = tfidf_matrix[mask]
    
    print(f"✅ Data aligned! New DF={len(df)}, TF-IDF={tfidf_matrix.shape}")
else:
    print("✅ Shapes are already aligned.")
# ==============================================

# Tampilkan dalam tabel
info_df = pd.DataFrame({
    'Komponen': ['Total Wisata', 'TF-IDF Matrix', 'IndoBERT Embeddings', 'Vocabulary Size'],
    'Nilai': [len(df), str(tfidf_matrix.shape), str(indobert_embeddings.shape), len(tfidf_vectorizer.vocabulary_)]
})
print("TABEL: DATA YANG DIMUAT")
display(info_df)

🔄 Loading data...

⚠️ SHAPE MISMATCH DETECTED: DF=296, IndoBERT=295
🔄 Aligning data by filtering empty descriptions (same as notebook 03)...
✅ Data aligned! New DF=295, TF-IDF=(295, 5000)
TABEL: DATA YANG DIMUAT


,Komponen,Nilai
0,Total Wisata,295
1,TF-IDF Matrix,"(295, 5000)"
2,IndoBERT Embeddings,"(295, 768)"
3,Vocabulary Size,5000


## 4.1 Hitung Similarity Matrix (Terpisah)

In [18]:
# Hitung similarity matrix untuk kedua metode secara terpisah
print("🔄 Computing similarity matrices...\n")

# PATH 1: N-gram + TF-IDF similarity (Untuk Rekomendasi/Related Items)
ngram_similarity = cosine_similarity(tfidf_matrix)

# PATH 2: IndoBERT similarity (Untuk Pencarian/Search)
indobert_similarity = cosine_similarity(indobert_embeddings)

# Combined (untuk evaluasi)
combined_similarity = 0.5 * ngram_similarity + 0.5 * indobert_similarity

# Simpan matriks
np.save(f'{DATA_PATH}ngram_similarity.npy', ngram_similarity)
np.save(f'{DATA_PATH}indobert_similarity.npy', indobert_similarity)
np.save(f'{DATA_PATH}combined_similarity.npy', combined_similarity)

# Tampilkan info
sim_info = pd.DataFrame({
    'Metode': ['N-gram + TF-IDF', 'IndoBERT', 'Combined'],
    'Tujuan': ['Rekomendasi (Detail Page)', 'Pencarian (Search Bar)', 'Evaluasi'],
    'Tipe': ['Lexical (Kata Kunci)', 'Semantic (Makna)', 'Hybrid'],
    'Matrix Shape': [str(ngram_similarity.shape), str(indobert_similarity.shape), str(combined_similarity.shape)]
})
print("TABEL: METODE SIMILARITY")
display(sim_info)

🔄 Computing similarity matrices...

TABEL: METODE SIMILARITY


,Metode,Tujuan,Tipe,Matrix Shape
0,N-gram + TF-IDF,Rekomendasi (Detail Page),Lexical (Kata Kunci),"(295, 295)"
1,IndoBERT,Pencarian (Search Bar),Semantic (Makna),"(295, 295)"
2,Combined,Evaluasi,Hybrid,"(295, 295)"


## 4.2 Load IndoBERT Model

In [19]:
# Load IndoBERT untuk encode user input saat SEARCH
MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def get_embedding(text):
    encoded = tokenizer(str(text), padding=True, truncation=True, max_length=128, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model(**encoded)
        embedding = mean_pooling(output, encoded['attention_mask'])
    return embedding.cpu().numpy()

print("✅ IndoBERT encoder ready!")

✅ IndoBERT encoder ready!


## 4.3 Fungsi Pencarian & Rekomendasi (Terpisah)

In [20]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def search_places(query, top_n=5):
    """
    FUNGSI 1: PENCARIAN (SEARCH)
    Menggunakan IndoBERT Embedding untuk mencari wisata berdasarkan makna query.
    """
    query_clean = preprocess_text(query)
    
    # Encode query -> IndoBERT Embedding
    query_embedding = get_embedding(query_clean)
    
    # Hitung cosine similarity dengan semua place embeddings
    bert_scores = cosine_similarity(query_embedding.reshape(1, -1), indobert_embeddings)[0]
    
    # Get top results
    top_indices = np.argsort(bert_scores)[::-1][:top_n]
    
    print(f"\nSEARCH QUERY (IndoBERT): {query}")
    print("="*100)
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        row = df.iloc[idx]
        results.append({
            'Rank': rank,
            'Nama Wisata': row['nama'],
            'Kategori': row['kategori'],
            'Score': round(bert_scores[idx], 4)
        })
    
    result_df = pd.DataFrame(results)
    display(result_df)
    return result_df

def get_detail_recommendations(place_name, top_n=5):
    """
    FUNGSI 2: REKOMENDASI DETAIL PAGE
    Menggunakan N-gram Similarity untuk mencari wisata serupa (keyword-based).
    """
    # Cari index tempat berdasarkan nama
    matches = df[df['nama'].str.lower() == place_name.lower()]
    if len(matches) == 0:
        print(f"❌ Tempat '{place_name}' tidak ditemukan.")
        return
    
    place_idx = matches.index[0]
    
    # Ambil skor similarity dari matriks N-gram
    sim_scores = ngram_similarity[place_idx]
    
    # Sort (exclude diri sendiri)
    top_indices = np.argsort(sim_scores)[::-1][1:top_n+1]
    
    print(f"\nREKOMENDASI UNTUK (N-gram): {place_name}")
    print("="*100)
    
    results = []
    for rank, idx in enumerate(top_indices, 1):
        row = df.iloc[idx]
        results.append({
            'Rank': rank,
            'Nama Wisata': row['nama'],
            'Kategori': row['kategori'],
            'Score': round(sim_scores[idx], 4)
        })
    
    result_df = pd.DataFrame(results)
    display(result_df)
    return result_df

## 4.4 Pengujian

In [ ]:
# TEST 1: Pencarian (pencarian teks bebas)
search_places("wisata air terjun yang sejuk", top_n=5)

# TEST 2: Rekomendasi (berdasarkan item)
sample_place = df['nama'].iloc[10]
get_detail_recommendations(sample_place, top_n=5)


SEARCH QUERY (IndoBERT): wisata air terjun yang sejuk


,Rank,Nama Wisata,Kategori,Score
0,1,Curug Country Cariu,Alam,0.2883
1,2,Gunung Peyek,Arena,0.2521
2,3,Curug Jengkol,Alam,0.2182
3,4,Green Canyon Citamiang,Alam,0.2169
4,5,Air Terjun Curug Kawung,Alam,0.1982



REKOMENDASI UNTUK (N-gram): Puncak Lalana


,Rank,Nama Wisata,Kategori,Score
0,1,Puncak Palasari,Arena,0.3038
1,2,Gunung Kencana,Olahraga,0.2396
2,3,Hiking Gunung Batu Jonggol,Olahraga,0.2196
3,4,Gunung Munara,Alam,0.2069
4,5,Bukit Alas Bandawasa,Arena,0.1987


,Rank,Nama Wisata,Kategori,Score
0,1,Puncak Palasari,Arena,0.3038
1,2,Gunung Kencana,Olahraga,0.2396
2,3,Hiking Gunung Batu Jonggol,Olahraga,0.2196
3,4,Gunung Munara,Alam,0.2069
4,5,Bukit Alas Bandawasa,Arena,0.1987


: 